In [2]:
%pip install pandas geopandas plotly scikit-learn numpy
import pandas as pd
import geopandas as gpd
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import numpy as np
import sqlite3


[notice] A new release of pip available: 22.2.2 -> 25.0
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
def load_and_transform_energy_data():
    # Read CSV with proper data types
    df = pd.read_csv('energy-and-utilities-linc.csv',
                     delimiter=';',
                     names=['County', 'Empty', 'Year', 'Variable', 'Value'],
                     dtype={'County': str, 'Empty': str, 'Year': str, 'Variable': str, 'Value': str})
    
    # Remove header rows and clean data
    df = df[~df['County'].isin(['Area Name', 'County'])]
    df = df[df['Value'].str.isnumeric().fillna(False)]

    # Drop empty column and convert types
    df = df.drop('Empty', axis=1)
    df['Value'] = pd.to_numeric(df['Value'])
    df['Year'] = pd.to_numeric(df['Year'])
    
    # Create pivot table
    transformed_df = pd.pivot_table(
        df,
        values='Value',
        index=['County', 'Year'],
        columns='Variable',
        aggfunc='first'
    ).reset_index()
    
    # Rename columns for clarity
    column_mapping = {
        'Occupied Housing Units Heated by Electricity': 'heated_by_electricity',
        'Occ Housing Units Heated by Gas Piped Underground': 'heated_by_gas',
        'Occupied Housing Units Heated by Fuel Oil': 'heated_by_fuel_oil',
        'Occ Housing Units Heated by Coal, Wood, Solar, or Other': 'heated_by_other',
        'Occupied Housing Units without House Heating Fuel': 'no_heating',
        'Occ Housing Units Heated by Bottled Tank or LP Gas Fuel': 'heated_by_lp_gas'
    }
    
    # Convert types and fill nulls
    transformed_df = transformed_df.rename(columns=column_mapping)
    transformed_df = transformed_df.fillna(0)
    
     # Convert to integers after cleaning
    numeric_cols = transformed_df.columns.difference(['County'])
    transformed_df[numeric_cols] = transformed_df[numeric_cols].astype(int)
    

    return transformed_df

# Cell 4: Create SQLite Database
def create_database():
    # Get transformed data
    energy_df = load_and_transform_energy_data()
    
    # Create database connection
    conn = sqlite3.connect('nc_energy.db')
    
    # Create table schema
    create_table_sql = '''
    CREATE TABLE IF NOT EXISTS energy_consumption (
        County TEXT,
        Year INTEGER,
        heated_by_electricity INTEGER,
        heated_by_gas INTEGER,
        heated_by_fuel_oil INTEGER,
        heated_by_other INTEGER,
        no_heating INTEGER,
        heated_by_lp_gas INTEGER,
        PRIMARY KEY (County, Year)
    ) WITHOUT ROWID;
    '''
    conn.executescript(create_table_sql)
    energy_df.to_sql('energy_consumption', conn, if_exists='replace', index=False)
    # Insert data
    energy_df.to_sql('energy_consumption', 
                     conn, 
                     if_exists='replace', 
                     index=False,
                     dtype={
                         'County': 'TEXT',
                         'Year': 'INTEGER',
                         'heated_by_electricity': 'INTEGER',
                         'heated_by_gas': 'INTEGER',
                         'heated_by_fuel_oil': 'INTEGER',
                         'heated_by_other': 'INTEGER',
                         'no_heating': 'INTEGER',
                         'heated_by_lp_gas': 'INTEGER'
                     })
    
    return conn, energy_df

def get_county_data(conn, county_name):
    query = '''
    SELECT 
        County,
        Year,
        heated_by_electricity,
        heated_by_gas,
        heated_by_fuel_oil,
        heated_by_other,
        no_heating,
        heated_by_lp_gas
    FROM energy_consumption 
    WHERE County = ?
    ORDER BY Year;
    '''
    return pd.read_sql(query, conn, params=(county_name,))

# Create database with new structure
energy_df = load_and_transform_energy_data()
print(energy_df.head())
print("\nColumns:", energy_df.columns.tolist())
print("\nData types:", energy_df.dtypes)

#Test get_county_data
conn, energy_df = create_database()
county_data = get_county_data(conn, 'Guilford County')
print(county_data.head())


Variable           County  Year  heated_by_lp_gas  heated_by_other  \
0         Alamance County  1990              4006             2541   
1         Alamance County  2000              6511             1005   
2         Alamance County  2010              6194             1243   
3         Alamance County  2015              5119             1918   
4         Alamance County  2020              4005             1101   

Variable  heated_by_gas  heated_by_electricity  heated_by_fuel_oil  no_heating  
0                 15946                  12543                7552          64  
1                 24211                  16783                2996          78  
2                 26390                  23032                2086          55  
3                 26122                  26576                1578         232  
4                 26842                  32290                 848         369  

Columns: ['County', 'Year', 'heated_by_lp_gas', 'heated_by_other', 'heated_by_gas', 'heated_

In [4]:
# Cell 3: Load Geographic Data
def load_geo_data():
    """Load and process geographic data"""
    counties_gdf = pd.read_csv('NCCountyCoordinates.csv')
    return counties_gdf
    
counties_gdf = load_geo_data()

In [14]:
# Cell 5: Query Functions
def get_county_data(conn, county_name):
    query = '''
    SELECT * FROM energy_consumption 
    WHERE county = ? 
    ORDER BY Year
    '''
    return pd.read_sql_query(query, conn, params=[county_name])

def get_county_list(conn):
    query = '''
    SELECT DISTINCT county FROM energy_consumption
    '''
    return pd.read_sql_query(query, conn)

#Print the returned queries
print(get_county_list(conn))
print(get_county_data(conn, 'Guilford County').head())

              County
0    Alamance County
1   Alexander County
2   Alleghany County
3       Anson County
4        Ashe County
..               ...
95      Wayne County
96     Wilkes County
97     Wilson County
98     Yadkin County
99     Yancey County

[100 rows x 1 columns]
            County  Year  heated_by_lp_gas  heated_by_other  heated_by_gas  \
0  Guilford County  1990              4291             5438          48770   
1  Guilford County  2000              7941             2034          76608   
2  Guilford County  2010              7944             1771          85699   
3  Guilford County  2015              6445             1999          82119   
4  Guilford County  2020              6132             1926          85206   

   heated_by_electricity  heated_by_fuel_oil  no_heating  
0                  57535               21482         190  
1                  71024               10715         345  
2                  87088                6580         479  
3                 1

In [24]:
def calculate_growth_patterns(df, heating_type):
    """Calculate historical growth patterns"""
    df = df.sort_values('Year')
    # Calculate year-over-year changes
    df['yearly_change'] = df[heating_type].diff()
    # Calculate percentage changes
    df['pct_change'] = df[heating_type].pct_change()
    # Calculate average growth rate
    df['avg_growth'] = df['pct_change'].rolling(window=2).mean()
    return df

def train_time_series_model(conn, county_name, heating_type):
    """Train model with historical patterns"""
    # Get historical data
    df = get_county_data(conn, county_name)
    df = calculate_growth_patterns(df, heating_type)
    
    # Calculate overall trend
    avg_growth_rate = df['pct_change'].mean()
    last_value = df[heating_type].iloc[-1]
    
    return df, avg_growth_rate, last_value

def predict_with_trends(df, avg_growth_rate, last_value, county_name, heating_type):
    """Generate predictions using historical trends"""
    future_years = range(2025, 2041, 5)
    predictions = []
    current_value = last_value
    
    for year in future_years:
        # Apply growth rate to previous value
        current_value = current_value * (1 + avg_growth_rate)
        
        predictions.append({
            'County': county_name,
            'Year': year,
            heating_type: int(current_value)
        })
    
    return pd.DataFrame(predictions)

# Usage
county_name = 'Guilford County'
heating_type = 'heated_by_electricity'

# Train model and get predictions
historical_data, growth_rate, last_value = train_time_series_model(conn, county_name, heating_type)
predictions = predict_with_trends(historical_data, growth_rate, last_value, county_name, heating_type)

# Print results
print(f"\nHistorical Growth Rate: {growth_rate:.2%}")
print(f"\nPredictions for {county_name}:")
print(predictions)


Historical Growth Rate: 18.20%

Predictions for Guilford County:
            County  Year  heated_by_electricity
0  Guilford County  2025                 131869
1  Guilford County  2030                 155873
2  Guilford County  2035                 184245
3  Guilford County  2040                 217782


In [20]:
# Cell 8: Visualization
def create_choropleth_map(conn, counties_gdf, heating_type, year):
    query = f"""
    SELECT County, Year, {heating_type}
    FROM energy_consumption
    WHERE Year = ?
    """
    df = pd.read_sql_query(query, conn, params=[year])
    
    fig = px.choropleth(
        df,
        geojson=counties_gdf,
        locations='County',
        featureidkey='properties.NAME',
        color=heating_type,
        scope="usa",
        title=f'NC {heating_type} by County ({year})',
        labels={heating_type: 'Number of Houses'}
    )
    
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        center={"lat": 35.5, "lon": -80},
        scope='usa',
    )
    
    return fig

# Create visualization
fig = create_choropleth_map(conn, counties_gdf, heating_type, 2020)
fig.show()

TypeError: Object of type DataFrame is not JSON serializable